In [1]:
import pandas as pd

In [2]:
# cargar dataset
data_path = "../datasets/df_final.csv"
df = pd.read_csv(data_path)

In [3]:
df.info() # Mostrar las primeras filas del DataFrame

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8882 entries, 0 to 8881
Data columns (total 60 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id_str                   8882 non-null   int64  
 1   date                     8882 non-null   object 
 2   rawContent               8882 non-null   object 
 3   replyCount               8882 non-null   int64  
 4   retweetCount             8882 non-null   int64  
 5   likeCount                8882 non-null   float64
 6   quoteCount               8882 non-null   int64  
 7   conversationId           8882 non-null   int64  
 8   hashtags                 8882 non-null   object 
 9   viewCount                8882 non-null   float64
 10  pcia                     8882 non-null   object 
 11  mes                      8882 non-null   int64  
 12  user_id                  8882 non-null   int64  
 13  pcia1                    8882 non-null   object 
 14  hashtags1               

In [4]:
from sentence_transformers import SentenceTransformer


# Modelo ligero y rápido, optimizado para español
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Para tus comentarios limpios
embeddings = model.encode(df.rawContent_clean.tolist(), 
                         batch_size=32,
                         show_progress_bar=True)

# add embeddings to DataFrame
df['embeddings'] = embeddings.tolist()

/home/ezequiel/ESTUDIO/estudio/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 278/278 [00:03<00:00, 86.61it/s] 


In [5]:
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import textwrap
import numpy as np

# --- 1. Preparación de datos ---
# Función para formatear el texto del hover
def wrap_text_for_plotly(text, width=40):
    return '<br>'.join(textwrap.wrap(text, width=width))

# Aplicar la función y crear una nueva columna para el texto del hover
df['texto_hover'] = df['rawContent_clean'].apply(wrap_text_for_plotly)


# --- 2. Reducción de Dimensión con PCA ---
# Extraer embeddings y aplicar PCA
pca = PCA(n_components=3)
embeddings_matrix = np.vstack(df['embeddings'].values)
datos_3d = pca.fit_transform(embeddings_matrix)

# Agregar los resultados de PCA como nuevas columnas al DataFrame
df['pca_x'] = datos_3d[:, 0]
df['pca_y'] = datos_3d[:, 1]
df['pca_z'] = datos_3d[:, 2]


# --- 3. Clustering con K-Means ---
k = 5
# Nota: n_init='auto' es la opción recomendada en versiones recientes de scikit-learn
kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
df['cluster'] = kmeans.fit_predict(datos_3d)


# --- 4. Visualización con Plotly ---
fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=df['pca_x'],
    y=df['pca_y'],
    z=df['pca_z'],
    mode='markers',
    marker=dict(
        color=df['cluster'],
        colorscale='Viridis',
        size=8, # Ajustado ligeramente para mejor visualización
        opacity=0.8
    ),
    hovertext=df['texto_hover'],
    hoverinfo='text'
))

fig.update_layout(
    title=f'Visualización de Clústeres de Comentarios (K={k})',
    scene=dict(
        xaxis_title='Componente PCA 1',
        yaxis_title='Componente PCA 2',
        zaxis_title='Componente PCA 3'
    ),
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()